In [1]:
import re
from datasets import load_dataset

# Load the dataset
dataset = load_dataset("Maxscha/commitbench")

# Enhanced cleaning function for the 'message' column
def clean_message(message, seen_messages):
    # Remove patterns like #<I>, <I>, HTML-like tags
    message = re.sub(r'#<I>\s*<I>', '', message)
    message = re.sub(r'<[^>]+>', '', message)

    # Remove specific patterns for issues and tasks
    message = re.sub(r'\(#.*?\)', '', message)  # Remove issue references in parentheses
    message = re.sub(r'closes #\d+', '', message)  # Remove "closes #" patterns
    message = re.sub(r'task\s*#:\s*', '', message, flags=re.IGNORECASE)  # Remove task references

    # Remove redundant punctuation and extra symbols
    message = re.sub(r'\*+', '', message)  # Remove excessive asterisks
    message = re.sub(r'-+', ' ', message)  # Replace multiple dashes with a space

    # Normalize spaces around punctuation
    message = re.sub(r'\s+', ' ', message).strip()  # Remove extra spaces

    # Normalize specific terms
    message = re.sub(r'\bnon empty\b', 'non-empty', message)  # Standardize "non empty" to "non-empty"
    message = re.sub(r'\bsetup\. toml\b', 'setup.toml', message)  # Standardize "setup. toml" to "setup.toml"
    message = re.sub(r'\bsetup\. py\b', 'setup.py', message)  # Standardize "setup. py" to "setup.py"

    # Fully spell out abbreviated terms (example)
    message = re.sub(r'\bcomput\.\.\.\b', 'computation of constructors', message)  # Example of expanding an abbreviation

    # Check for incomplete phrases and replace or clarify if necessary
    if message == "bumpy to":
        message = "description unclear, please clarify"  # Replace with a clarifying statement

    # Check for duplicates and consolidate messages
    if message in seen_messages:
        return None  # Return None for duplicate messages
    seen_messages.add(message)  # Add the message to the seen set

    # Lowercase the message
    return message.lower()

# Set to track seen messages
seen_messages = set()

# Apply the cleaning function to the 'message' column for all splits
cleaned_dataset = dataset.map(lambda x: {"message": clean_message(x["message"], seen_messages)}, batched=False)

# Filter out None values (duplicates or invalid messages)
cleaned_dataset = cleaned_dataset.filter(lambda x: x["message"] is not None)

In [2]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("Salesforce/codet5-small")

MAX_DIFF_LENGTH = 512  # For the 'diff' input
MAX_MESSAGE_LENGTH = 128  # For the 'message' output

# Tokenize the dataset (with separate max lengths for diff and message)
def tokenize_function(examples):
    # Tokenize the diffs as input with a max length of 512
    model_inputs = tokenizer(
        examples["diff"], 
        padding="max_length", 
        truncation=True, 
        max_length=MAX_DIFF_LENGTH  # Max length for diff
    )
    
    # Tokenize the commit messages as target/output labels with a max length of 128
    labels = tokenizer(
        text_target=examples["message"], 
        padding="max_length", 
        truncation=True, 
        max_length=MAX_MESSAGE_LENGTH  # Max length for message
    )
    
    # Assign tokenized commit messages to the 'labels' key
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply the tokenization to the cleaned dataset
tokenized_datasets = cleaned_dataset.map(tokenize_function, batched=True)

In [3]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

# Define training arguments
training_args = TrainingArguments(
    output_dir="./teamspace/uploads/result",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=70,  # Reduced to fit L4 VRAM
    per_device_eval_batch_size=70,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_dir='./teamspace/uploads/logs',
    logging_steps=1000,
    save_total_limit=1,
    save_strategy="epoch",
    load_best_model_at_end=True,
    run_name="swiftcommit-finetuned",
    fp16=True,
    optim="adamw_torch", 
    dataloader_num_workers=2,
)

# Fix tokenizer deadlocks
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [4]:
# Set up the trainer with early stopping callback
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]  # Adjust patience as needed
)

trainer.train()

/tmp/ipykernel_108679/3208162545.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,0.478100,0.452124
2,0.460300,0.436395
3,0.447600,0.427601
4,0.438900,0.422449
5,0.431500,0.417885
6,0.427700,0.415021
7,0.422500,0.412981
8,0.418100,0.411546
9,0.416600,0.410703
10,0.415300,0.410322


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=160570, training_loss=0.4399933359279515, metrics={'train_runtime': 186883.2015, 'train_samples_per_second': 60.141, 'train_steps_per_second': 0.859, 'total_flos': 1.5211538763743232e+18, 'train_loss': 0.4399933359279515, 'epoch': 10.0})

In [11]:
model.save_pretrained("teamspace/trans_codeT5/model")
tokenizer.save_pretrained("teamspace/trans_codeT5/tokenizer")

('teamspace/trans_codeT5/tokenizer/tokenizer_config.json',
 'teamspace/trans_codeT5/tokenizer/special_tokens_map.json',
 'teamspace/trans_codeT5/tokenizer/vocab.json',
 'teamspace/trans_codeT5/tokenizer/merges.txt',
 'teamspace/trans_codeT5/tokenizer/added_tokens.json',
 'teamspace/trans_codeT5/tokenizer/tokenizer.json')

In [13]:
ls teamspace/trans_codeT5

config.json             model.safetensors        tokenizer_config.json
generation_config.json  special_tokens_map.json  vocab.json
merges.txt              tokenizer/
model/                  tokenizer.json
